# Stage 3 Launcher (V2 Retrain on Dropout-Trained Backbone)

**What this does:** loads the encoder + NER weights from Stage 1's `best_cls_only.pt` (CLS-only NER F1=0.5841), freezes them, and trains a **fresh V2 cross-attention sentiment head** from Xavier init for 10 epochs.

**Why Stage 1's checkpoint, not Stage 2's?**
- Stage 3 discards the sentiment head from its input (re-initializes V2 fresh)
- Only the encoder + NER weights survive the load
- Stage 1's checkpoint has CLS-only NER F1=0.5841 vs Stage 2's 0.5335 — better encoder for the e2e inference regime

**Why NO global-attn dropout here?**
- V2 sentiment head only ever sees entity-aware encoder outputs at inference (Pass 2 of e2e uses `CLS + predicted_entity_tokens` global attention)
- Adding CLS-only regime to V2's training distribution would waste capacity on a regime V2 never encounters at inference
- The encoder is frozen, so its CLS-only robustness (already baked in from Stage 1's dropout training) is preserved automatically — V2 just needs to learn entity-aware sentiment extraction

**Estimated time:** ~2-3 hours on the 95 GB GPU at batch_size=80 (10 epochs of 17K articles).

**Target metric:** **Pearson r ≥ 0.50** on holdout (the Apr-18 V2 baseline). Sentiment MSE ≤ 0.08 on val (gold-mask regime).

In [ ]:
# 1. Mount Drive & check GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path & locate Stage 1 checkpoint
import os
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/training/train_stage3.py"), "train_stage3.py not found!"

# Source: Stage 1's CLS-only-best checkpoint (best encoder + NER for e2e regime)
SOURCE_CKPT = f"{PROJECT_PATH}/checkpoints/stage1_ner_large_v2/best_cls_only.pt"
assert os.path.exists(SOURCE_CKPT), f"Source checkpoint missing: {SOURCE_CKPT}"
src_gb = os.path.getsize(SOURCE_CKPT) / 1e9
print(f"Project       : {PROJECT_PATH}")
print(f"Source ckpt   : {SOURCE_CKPT}  ({src_gb:.2f} GB)")

# Local-first save
LOCAL_CKPT_DIR = "/content/stage3_local"
# New Drive dir — DO NOT overwrite the Apr-18 Stage 3 results (preserved in archive/)
DRIVE_CKPT_DIR = f"{PROJECT_PATH}/checkpoints/stage3_sentiment_large_v2retrain"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"Local ckpts   : {LOCAL_CKPT_DIR}")
print(f"Drive ckpts   : {DRIVE_CKPT_DIR}")

In [ ]:
# 3. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. Run Stage 3 V2 retrain
#    Key flags:
#      --source-checkpoint           ← Stage 1's best_cls_only.pt (not the old Apr-18 Stage 2)
#      --drive-ckpt-dir              ← new dir to preserve Apr-18 results
#      --batch-size 80               ← G4 95GB can comfortably handle ~80 (Apr-18 used 50 on 80GB)
#      --epochs 10                   ← same as Apr-18 baseline
#      --lr 5e-4                     ← high LR for the tiny ~5M trainable params
#
#    Notes:
#    - Encoder + NER are frozen. Only V2 sentiment head trains.
#    - V2 is initialized fresh (Xavier) — the V2 weights in Stage 1's checkpoint are
#      discarded and re-init'd from scratch.
#    - No global-attn dropout (V2 only sees entity-aware encoder at inference).
#
#    Watch: per-epoch "sent_corr=..." — this is the Pearson r metric on val. Want it
#    to climb above 0.50 by epoch 8-10 (Apr-18 baseline reached 0.4983).

!cd {PROJECT_PATH} && python scripts/training/train_stage3.py \
    --source-checkpoint {SOURCE_CKPT} \
    --drive-ckpt-dir {DRIVE_CKPT_DIR} \
    --local-ckpt-dir {LOCAL_CKPT_DIR} \
    --batch-size 80 \
    --epochs 10 \
    --lr 5e-4 \
    --patience 3

In [ ]:
# 5. Show local checkpoints + tail of training log
#    train_stage3.py already does Drive sync internally, but we'll verify here.
import os, glob
print(f"Local checkpoints in {LOCAL_CKPT_DIR}:")
for f in sorted(os.listdir(LOCAL_CKPT_DIR)):
    path = os.path.join(LOCAL_CKPT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f:40s} {size:8.1f} MB")

# Local log (the script writes a stage3_sentiment_<ts>.log to LOCAL_CKPT_DIR)
local_logs = sorted(glob.glob(f"{LOCAL_CKPT_DIR}/stage3_sentiment_*.log"))
if local_logs:
    print(f"\nLatest local log: {local_logs[-1]} (last 40 lines)")
    with open(local_logs[-1]) as fh:
        for line in fh.readlines()[-40:]:
            print(f"  {line.rstrip()}")

In [ ]:
# 6. Verify Drive sync (train_stage3.py handles its own sync, but double-check)
#    The script's final step is flush_and_unmount + remount, so Drive may need a
#    moment after the script exits.
import os, time, shutil

# Re-mount Drive in case the script unmounted at the end
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    time.sleep(5)

print(f"Drive checkpoints in {DRIVE_CKPT_DIR}:")
if os.path.exists(DRIVE_CKPT_DIR):
    for f in sorted(os.listdir(DRIVE_CKPT_DIR)):
        path = os.path.join(DRIVE_CKPT_DIR, f)
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1e6
            print(f"  {f:40s} {size:8.1f} MB")
else:
    print(f"  Directory not visible — Drive may need more time.")

# Backup: if any .pt files are still only in /content, copy them now
print(f"\nBackup sync (in case train_stage3.py's sync missed anything):")
for f in sorted(os.listdir(LOCAL_CKPT_DIR)):
    src = os.path.join(LOCAL_CKPT_DIR, f)
    dst = os.path.join(DRIVE_CKPT_DIR, f)
    if os.path.isfile(src) and not os.path.exists(dst):
        try:
            shutil.copy2(src, dst)
            print(f"  copied {f}")
        except Exception as e:
            print(f"  copy failed: {f} - {e}")

In [ ]:
# 7. (Optional) Terminate runtime to stop billing
#    Only run after cell 6 confirms checkpoints are on Drive.
from google.colab import runtime
runtime.unassign()